In [0]:
%run ../utils

In [0]:
# 2_2_02 — email body child, RE-SOURCED Nov 12 2026 from the SILVER PARENT.
# The old path re-parsed BRONZE per version and appended one row per VERSION of
# every email activity — the root cause of this child's duplication (367K+ rows
# vs one row per email activity). Now parent-driven Option B CDC, exactly like
# siblings 2_2_03/04/05: watermark on THIS child, changed parents only, LATEST
# version per id, exact DELETE of affected ids, then engine write_silver.
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

CHILD = 'lead_activites_email_body'
PARENT = 'crm_ingestion.silver.lead_activites_processed'
try:
    watermark = get_silver_watermark(CHILD)
except Exception:
    watermark = None

_parent_df = spark.table(PARENT)
has_body = 'body_text_quoted' in _parent_df.columns
cols = ['id', 'lead_id', 'activity_at', 'type', 'status', 'date_updated',
        'bronze_insert_date', 'bronze_updated_at'] + (['body_text_quoted'] if has_body else [])
src = _parent_df.select(*cols)

if watermark is not None:
    wm_str = watermark.strftime('%Y-%m-%d %H:%M:%S.%f')
    print(f"CDC MODE: parents updated after {wm_str}")
    src = src.filter(col('bronze_updated_at') > watermark)
else:
    print("FULL LOAD: all activities, latest version per id")

# LATEST version per id — parity with 2_2_03/04/05; a no-op once the parent
# is one-row-per-id (engine rebuild / CDC MERGE contract)
_version = Window.partitionBy('id').orderBy(col('bronze_updated_at').desc(), col('bronze_insert_date').desc())
df_in = src.withColumn('_v', row_number().over(_version)).filter(col('_v') == 1).drop('_v')

SKIP = (not has_body) or (watermark is not None and len(df_in.take(1)) == 0)
if not has_body:
    print("⏭️ SKIP: parent schema has no body_text_quoted (no email activities in history)")
elif SKIP:
    print(f"✅ CDC: No changed activities since {watermark}")
else:
    print(f"body_text_quoted in parent? {has_body}")

In [0]:
if SKIP:
    df_email_body = None
else:
    # FRAGMENT FIX (Nov 12 2026): the original extraction kept ONLY fragment 0
    # (body_text_quoted[0]["text"]) — every activity with 2+ body fragments lost
    # the later fragments in this child (measured: 102 activities where the
    # Calendly 'A new event has been scheduled.' phrase sits in the second
    # fragment / quoted reply history, silently breaking phrase-based gold
    # consumers). One row per FRAGMENT now, same schema — this child's write
    # contract is multi-row per activity (legacy delete+append, no pk
    # collapse). A one-time backfill (Nov 12 2026) inserted the previously
    # missing fragment rows into the live table.
    from pyspark.sql.functions import explode
    df_email_body = (
        df_in.filter(col("type") == "Email")
        .select(
            col("bronze_insert_date"),
            col("bronze_updated_at"),
            col("id").alias("activity_id"),
            col("lead_id"),
            col("activity_at"),
            col("status"),
            col("date_updated"),
            explode("body_text_quoted").alias("fragment"),
        )
        .select(
            col("bronze_insert_date"),
            col("bronze_updated_at"),
            col("activity_id"),
            col("lead_id"),
            col("activity_at"),
            col("status"),
            col("date_updated"),
            col("fragment")["text"].alias("body_text"),
        )
        .filter(col("body_text").isNotNull())
    )
    print(f"email activities (latest version each, ONE ROW PER BODY FRAGMENT): {df_email_body.count():,} planned rows")

In [0]:
if df_email_body is not None:
    if watermark is not None:
        # Exact DELETE of affected activity ids — a newer version of the SAME
        # activity carries a newer bronze stamp, so write_silver's timestamp
        # window alone would leave the stale sibling row behind (duplicate ids).
        spark.sql(f"""
            DELETE FROM crm_ingestion.silver.{CHILD}
            WHERE activity_id IN (
                SELECT id FROM crm_ingestion.silver.lead_activites_processed
                WHERE bronze_updated_at > '{wm_str}'
            )
        """)
        _CDC_WATERMARK = watermark  # engine write_silver: CDC append path
    write_silver(df_email_body, CHILD)
    _CDC_WATERMARK = None
else:
    print("⏭️ Nothing to write — table unchanged")

In [0]:
display(spark.table('crm_ingestion.silver.lead_activites_email_body').limit(10))